In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats
from scipy.stats import anderson
import statsmodels.api as sm
import matplotlib.pyplot as plt
from ordered_set import OrderedSet
from sklearn.preprocessing import RobustScaler
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import copy
import re
import string
from langdetect import detect
from stop_words import safe_get_stop_words
from stop_words import get_stop_words
from nltk.tokenize import word_tokenize
import spacy
import torch

In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
with open("Data/language_acronyms_dict.pickle", "rb") as f:
    language_acronyms_dict = pickle.load(f)

In [4]:
df = pd.read_pickle("Data/preprocessed_not_imputed.pickle")

In [5]:
def load_language_models():
    language_models = {"ca": spacy.load("ca_core_news_sm"),
                   "el": spacy.load("el_core_news_sm"),
                   "en": spacy.load("en_core_web_sm"),
                   "es": spacy.load("es_core_news_sm"),
                   "it": spacy.load("it_core_news_sm"),
                   "nl": spacy.load("nl_core_news_sm"),
                   "pl": spacy.load("pl_core_news_sm"),
                   "sv": spacy.load("sv_core_news_sm")}
    return language_models

def retrieve_stop_words(language_acronyms_dict):
    stop_words_dict = {}

    for language_acronym in language_acronyms_dict.keys():
        functionality = language_acronyms_dict[language_acronym][1]
        language = language_acronyms_dict[language_acronym][0]
        
        if functionality == "supported":
            stop_words_dict[language_acronym] = get_stop_words(language.lower())
        else:
            continue
    return stop_words_dict

def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

def remove_stopwords_lemmatize(text: str, language: str, language_models, stop_words_dict, language_acronyms_dict = language_acronyms_dict):
    if language in stop_words_dict.keys():
        if language in language_models.keys(): #yes lemma
            #print("{} is in language models".format(language))
            if language_acronyms_dict[language][1] == "supported": #yes lemma, yes stopword
                nlp = language_models[language]
                stop_words = stop_words_dict.get(language, set())
                doc = nlp(text)
                clean_text_list = [token.lemma_ for token in doc if token.text.lower() not in stop_words]
                clean_text = ' '.join(clean_text_list)
                return clean_text
            else:                                                 #yes lemma, no stopword
                nlp = language_models[language]
                doc = nlp(text)
                clean_text_list = [token.lemma_ for token in doc]
                clean_text = ' '.join(clean_text_list)
                return clean_text
        else: #no lemma
            #print("{} is not in language models".format(language))
            if language_acronyms_dict[language][1] == "supported": #no lemma, yes stopwords
                stop_words = stop_words_dict.get(language, set())
                text_list = text.split()
                clean_text_list = [word for word in text_list if word not in stop_words]
                clean_text = ' '.join(clean_text_list)
                return clean_text
            else:                                                 #no lemma, no stopword
                return text
    else:                                                         #unknown language
        #print("{} is not known".format(language))
        return text

#let's preprocess the text columns by lowercasing, removing numbers and special characters, removing stop words and lemmatization
def clean_text(text, language):
    #to lowercase
    text = text.lower()
    #remove links
    pattern = r'https?://\S+|www\.\S+'
    text = re.sub(pattern, "", text)
    #remove punctuation
    pattern = r'[{}]'.format(re.escape(string.punctuation))
    text = re.sub(pattern, "", text)
    #remove next line
    pattern = r'[^ \w\.]'
    text = re.sub(pattern, "", text) 
    #remove words containing numbers
    pattern = r'\w*\d\w*'
    text = re.sub(pattern, '', text)
    text = remove_stopwords_lemmatize(text, language, language_models=language_models, stop_words_dict=stop_words_dict)
    return text

def apply_text_preprocessing(df, columns):
    for col in columns:
        for i in tqdm(range(0, len(df))):
            if pd.isna(df[col].iloc[i]) == False:
                text = df[col].iloc[i]
                language = df["language"].iloc[i]
                df.at[i, col] = clean_text(text, language)
    return df

def language_unknown(df):
    language_unknown = []
    for i in range(0, len(df)):
        if df["language"].iloc[i] == "unknown":
            language_unknown.append(i)
    df = df.drop(labels = language_unknown)
    print("{} amount of rows were removed because the language was unknown".format(len(language_unknown)))
    return df

In [6]:
#initialize variables
text_cols = ["short_descr", "object_contract/title", "object_descr/title", "ac_cost/ac_criterion", "ac_criterion"]

In [ ]:
#get the languages
df["language"] = df[text_cols[0]].apply(detect_language)
#load the language models
language_models = load_language_models()
#get stopwords dict
stop_words_dict = retrieve_stop_words(language_acronyms_dict = language_acronyms_dict)
#remove rows where language was not detected
df = language_unknown(df)

In [ ]:
#apply preprocessing
df = apply_text_preprocessing(df, text_cols)

In [10]:
with open("Data/df_preprocessed_13_11", "wb") as f:
    pickle.dump(df, f)